# Phase 2 — Pose Estimation
## Heatmap 원리 + OKS 평가 + YOLOv8-pose + 자율주행 응용

### 학습 목표
- **Heatmap 방식** 직접 구현 — keypoint 예측의 수학적 원리
- **OKS (Object Keypoint Similarity)** 직접 구현 — IoU의 keypoint 버전
- **YOLOv8-pose** 추론 + skeleton 시각화 — Phase 1 생태계 연장
- **자율주행 응용** — 보행자 몸 방향으로 횡단 의도 판단
- **mAP@OKS 평가** — 정량 지표 파이프라인 구축

### COCO 17 Keypoints
```
 0: nose          1: left_eye      2: right_eye
 3: left_ear      4: right_ear     5: left_shoulder
 6: right_shoulder 7: left_elbow   8: right_elbow
 9: left_wrist   10: right_wrist  11: left_hip
12: right_hip    13: left_knee    14: right_knee
15: left_ankle   16: right_ankle
```


## 0. 환경 확인


In [ ]:
import torch
import numpy as np
import cv2
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
print(f'OpenCV   : {cv2.__version__}')

# MediaPipe 설치 확인
try:
    import mediapipe as mp
    print(f'MediaPipe: {mp.__version__}')
except ImportError:
    print('MediaPipe: 미설치 → pip install mediapipe')


## 1. Heatmap 방식 — keypoint 예측 원리

### 왜 좌표를 직접 예측하지 않나?
```
직접 회귀(regression): 모델이 (x, y) 좌표 숫자를 바로 출력
  → gradient가 불안정, 작은 이미지 변화에 민감

Heatmap 방식: 각 keypoint마다 W×H 확률 맵 출력
  → 어디에 관절이 있을 확률이 높은지 분포로 표현
  → argmax로 좌표 추출 (sub-pixel: 가중 평균)
  → gradient 안정적, 공간 정보 보존
```

### Gaussian Heatmap 생성
keypoint 위치 `(x, y)` 중심으로 Gaussian 분포 생성:
$$H(i,j) = \exp\left(-\frac{(i-x)^2 + (j-y)^2}{2\sigma^2}\right)$$

- $\sigma$: 관절 크기/불확실성에 따라 조절 (큰 관절 = 큰 $\sigma$)
- 학습 시 GT heatmap과 예측 heatmap의 MSE를 최소화


In [ ]:
def generate_gaussian_heatmap(height, width, keypoints, sigma=6):
    """
    keypoints: [(x, y, visibility), ...]  17개
    visibility: 0=없음, 1=가려짐, 2=보임
    반환: (17, H, W) heatmap
    """
    num_kp = len(keypoints)
    heatmaps = np.zeros((num_kp, height, width), dtype=np.float32)

    # 미리 좌표 그리드 생성 (브로드캐스팅용)
    xs = np.arange(width)
    ys = np.arange(height)
    grid_x, grid_y = np.meshgrid(xs, ys)  # (H, W)

    for k, (x, y, vis) in enumerate(keypoints):
        if vis == 0:  # 이미지 밖 or 없는 keypoint
            continue
        # Gaussian: exp(-((i-x)² + (j-y)²) / 2σ²)
        dist_sq = (grid_x - x)**2 + (grid_y - y)**2
        heatmaps[k] = np.exp(-dist_sq / (2 * sigma**2))

    return heatmaps


def heatmap_to_coords(heatmaps):
    """
    Heatmap → (x, y) 좌표 추출 (argmax)
    heatmaps: (K, H, W)
    반환: (K, 2) 좌표
    """
    K, H, W = heatmaps.shape
    coords = np.zeros((K, 2))
    for k in range(K):
        idx = np.argmax(heatmaps[k])
        coords[k, 1] = idx // W  # y
        coords[k, 0] = idx % W   # x
    return coords


# ── 동작 확인: 가상 keypoint로 heatmap 생성 ──
H, W = 128, 128
# 단순화된 상체 keypoint (코, 왼어깨, 오른어깨, 왼팔꿈치, 오른팔꿈치)
sample_kps = [
    (64, 20, 2),   # 코
    (45, 40, 2),   # 왼어깨
    (83, 40, 2),   # 오른어깨
    (30, 65, 2),   # 왼팔꿈치
    (98, 65, 2),   # 오른팔꿈치
]

heatmaps = generate_gaussian_heatmap(H, W, sample_kps, sigma=6)
coords   = heatmap_to_coords(heatmaps)

print('Heatmap 생성 완료')
print(f'  shape: {heatmaps.shape}  (keypoint수, H, W)')
print('  복원된 좌표:')
kp_names = ['코', '왼어깨', '오른어깨', '왼팔꿈치', '오른팔꿈치']
for i, (name, (gx, gy, _)) in enumerate(zip(kp_names, sample_kps)):
    rx, ry = coords[i]
    print(f'    {name}: GT({gx},{gy}) → 복원({rx:.0f},{ry:.0f})')


In [ ]:
# ── Heatmap 시각화 ──
fig, axes = plt.subplots(1, 6, figsize=(18, 4))

# 합성 heatmap (모든 keypoint 합)
combined = heatmaps.sum(axis=0)
axes[0].imshow(combined, cmap='hot')
axes[0].set_title('합성 Heatmap', fontsize=10)
axes[0].axis('off')

for i, name in enumerate(kp_names):
    axes[i+1].imshow(heatmaps[i], cmap='hot', vmin=0, vmax=1)
    axes[i+1].scatter([sample_kps[i][0]], [sample_kps[i][1]],
                      c='cyan', s=50, zorder=5)
    axes[i+1].set_title(name, fontsize=10)
    axes[i+1].axis('off')

plt.suptitle('Gaussian Heatmap — 각 keypoint의 존재 확률 분포', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('heatmap_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('밝은 부분 = 관절이 있을 확률이 높은 위치')


## 2. OKS (Object Keypoint Similarity) — keypoint용 IoU

### 정의
Detection의 IoU처럼, Pose에서 예측과 GT의 유사도를 측정:

$$OKS = \frac{\sum_i \exp\left(-\frac{d_i^2}{2s^2k_i^2}\right) \cdot \delta(v_i > 0)}{\sum_i \delta(v_i > 0)}$$

| 기호 | 의미 |
|------|------|
| $d_i$ | keypoint $i$의 예측-GT 거리 (픽셀) |
| $s$ | 객체 크기 (bbox 면적의 제곱근) |
| $k_i$ | keypoint별 허용 오차 상수 (코 0.026, 무릎 0.072 등) |
| $v_i$ | GT visibility (0이면 평가 제외) |

**$k_i$의 의미**: 관절마다 허용 오차가 다름
- 눈/코: 작음 (정밀 위치가 중요)
- 엉덩이/무릎: 큼 (약간 틀려도 괜찮음)


In [ ]:
# COCO keypoint별 허용 오차 상수 (논문 기준)
COCO_SIGMAS = np.array([
    0.026,  # 0: nose
    0.025,  # 1: left_eye
    0.025,  # 2: right_eye
    0.035,  # 3: left_ear
    0.035,  # 4: right_ear
    0.079,  # 5: left_shoulder
    0.079,  # 6: right_shoulder
    0.072,  # 7: left_elbow
    0.072,  # 8: right_elbow
    0.062,  # 9: left_wrist
    0.062,  # 10: right_wrist
    0.107,  # 11: left_hip
    0.107,  # 12: right_hip
    0.087,  # 13: left_knee
    0.087,  # 14: right_knee
    0.089,  # 15: left_ankle
    0.089,  # 16: right_ankle
])


def compute_oks(pred_kps, gt_kps, bbox_area, sigmas=COCO_SIGMAS):
    """
    OKS 계산.
    pred_kps : (17, 2) 예측 keypoint (x, y)
    gt_kps   : (17, 3) GT keypoint (x, y, visibility)
    bbox_area: GT bounding box 면적 (s² 계산용)
    """
    s = np.sqrt(bbox_area)  # 객체 크기
    oks_sum = 0.0
    valid_count = 0

    for i in range(len(gt_kps)):
        vx, vy, vis = gt_kps[i]
        if vis == 0:
            continue
        dx = pred_kps[i, 0] - vx
        dy = pred_kps[i, 1] - vy
        d_sq = dx**2 + dy**2
        k_sq = (2 * sigmas[i])**2
        oks_sum += np.exp(-d_sq / (2 * s**2 * k_sq))
        valid_count += 1

    return oks_sum / valid_count if valid_count > 0 else 0.0


# ── OKS 동작 확인 ──
np.random.seed(42)

# GT keypoint (17개, COCO 형식)
gt_kps = np.array([
    [320, 80, 2],   # nose
    [310, 75, 2],   # left_eye
    [330, 75, 2],   # right_eye
    [300, 85, 1],   # left_ear (가려짐)
    [340, 85, 1],   # right_ear
    [290, 130, 2],  # left_shoulder
    [350, 130, 2],  # right_shoulder
    [270, 180, 2],  # left_elbow
    [370, 180, 2],  # right_elbow
    [255, 225, 2],  # left_wrist
    [385, 225, 2],  # right_wrist
    [295, 250, 2],  # left_hip
    [345, 250, 2],  # right_hip
    [290, 320, 2],  # left_knee
    [350, 320, 2],  # right_knee
    [285, 390, 2],  # left_ankle
    [355, 390, 2],  # right_ankle
], dtype=float)

bbox_area = 100 * 320  # 대략적인 사람 bbox 면적

# 시나리오 비교
scenarios = {
    '완벽한 예측 (오차 0px)':    gt_kps[:, :2].copy(),
    '좋은 예측 (오차 ~5px)':     gt_kps[:, :2] + np.random.uniform(-5, 5, (17, 2)),
    '보통 예측 (오차 ~20px)':    gt_kps[:, :2] + np.random.uniform(-20, 20, (17, 2)),
    '나쁜 예측 (오차 ~50px)':    gt_kps[:, :2] + np.random.uniform(-50, 50, (17, 2)),
}

print('OKS 계산 결과')
print('-' * 40)
for name, pred in scenarios.items():
    oks = compute_oks(pred, gt_kps, bbox_area)
    bar = '█' * int(oks * 20)
    print(f'{name:25s}: OKS={oks:.3f}  {bar}')
print()
print('OKS >= 0.5: 매칭으로 간주 (mAP@OKS=0.5)')
print('OKS >= 0.75: 엄격한 매칭 (mAP@OKS=0.75)')


## 3. YOLOv8-pose 추론 + Skeleton 시각화

YOLOv8는 Detection 외에 Pose 모델도 제공.  
Phase 1과 동일한 Ultralytics API → **아키텍처 일관성**

```
yolov8n-pose.pt  (nano,   ~3.3M params)
yolov8s-pose.pt  (small,  ~11.6M params)
yolov8m-pose.pt  (medium, ~26.4M params)
```

출력 형식: Detection bbox + 17개 keypoint (x, y, confidence)


In [ ]:
from ultralytics import YOLO
from pathlib import Path

# Pose 모델 로드 (첫 실행 시 자동 다운로드 ~6MB)
pose_model = YOLO('yolov8n-pose.pt')
print('YOLOv8n-pose 로드 완료')
print(f'파라미터 수: {sum(p.numel() for p in pose_model.model.parameters()):,}')


In [ ]:
# COCO128 이미지로 추론
data_path = Path.home() / 'MyProject' / 'datasets' / 'coco128'
img_paths = sorted((data_path / 'images' / 'train2017').glob('*.jpg'))

# 사람이 포함된 이미지 찾기
person_imgs = []
for p in img_paths:
    label_path = data_path / 'labels' / 'train2017' / (p.stem + '.txt')
    if label_path.exists():
        content = label_path.read_text().strip()
        if content and content.split()[0] == '0':  # class 0 = person
            person_imgs.append(p)
    if len(person_imgs) >= 9:
        break

print(f'사람 포함 이미지: {len(person_imgs)}장')


In [ ]:
# ── Skeleton 정의 (COCO 연결 관계) ──
SKELETON = [
    (0, 1), (0, 2),           # 코 → 눈
    (1, 3), (2, 4),           # 눈 → 귀
    (5, 6),                   # 어깨
    (5, 7), (7, 9),           # 왼팔
    (6, 8), (8, 10),          # 오른팔
    (5, 11), (6, 12),         # 어깨 → 엉덩이
    (11, 12),                 # 엉덩이
    (11, 13), (13, 15),       # 왼다리
    (12, 14), (14, 16),       # 오른다리
]

KP_COLORS = [
    (255, 0, 0),    # 0 nose
    (255, 85, 0),   # 1 left_eye
    (255, 170, 0),  # 2 right_eye
    (255, 255, 0),  # 3 left_ear
    (170, 255, 0),  # 4 right_ear
    (85, 255, 0),   # 5 left_shoulder
    (0, 255, 0),    # 6 right_shoulder
    (0, 255, 85),   # 7 left_elbow
    (0, 255, 170),  # 8 right_elbow
    (0, 255, 255),  # 9 left_wrist
    (0, 170, 255),  # 10 right_wrist
    (0, 85, 255),   # 11 left_hip
    (0, 0, 255),    # 12 right_hip
    (85, 0, 255),   # 13 left_knee
    (170, 0, 255),  # 14 right_knee
    (255, 0, 255),  # 15 left_ankle
    (255, 0, 170),  # 16 right_ankle
]


def draw_pose(img, keypoints, scores, kp_thresh=0.5):
    """
    img: BGR numpy array
    keypoints: (17, 2) xy 좌표
    scores: (17,) confidence
    """
    img = img.copy()
    # skeleton 연결선
    for (i, j) in SKELETON:
        if scores[i] > kp_thresh and scores[j] > kp_thresh:
            pt1 = tuple(map(int, keypoints[i]))
            pt2 = tuple(map(int, keypoints[j]))
            cv2.line(img, pt1, pt2, (200, 200, 200), 2)
    # keypoint 원
    for k in range(17):
        if scores[k] > kp_thresh:
            pt = tuple(map(int, keypoints[k]))
            cv2.circle(img, pt, 4, KP_COLORS[k], -1)
    return img


print('Skeleton 그리기 함수 정의 완료')


In [ ]:
# ── 추론 + 시각화 ──
n_show = min(6, len(person_imgs))
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i in range(n_show):
    img_bgr = cv2.imread(str(person_imgs[i]))
    result  = pose_model(person_imgs[i], conf=0.3, verbose=False)[0]

    vis = img_bgr.copy()

    if result.keypoints is not None and len(result.keypoints):
        kps_all    = result.keypoints.xy.cpu().numpy()    # (N, 17, 2)
        scores_all = result.keypoints.conf.cpu().numpy()  # (N, 17)

        for person_idx in range(len(kps_all)):
            vis = draw_pose(vis, kps_all[person_idx], scores_all[person_idx])

        n_persons = len(kps_all)
    else:
        n_persons = 0

    axes[i].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f'{person_imgs[i].name} | {n_persons}명', fontsize=9)
    axes[i].axis('off')

plt.suptitle('YOLOv8n-pose — Skeleton 시각화', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('pose_skeleton.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. 자율주행 응용 — 보행자 의도 예측

### 문제
자율주행 차량이 교차로에서 보행자를 발견했을 때:
- 보행자가 길을 **건너려는지** vs **서있는지** 판단 필요
- 카메라 한 장으로 판단해야 함

### Pose 기반 판단 방법
```
몸통 방향 (Torso Orientation):
  → 왼어깨/오른어깨의 x 좌표 차이
  → 어깨가 도로 방향으로 향하면 '횡단 의도 높음'

보폭 (Stride):
  → 왼발목/오른발목 y 좌표 차이
  → 발목이 수직으로 벌어지면 '걷는 중'

무게중심 (CoM, Center of Mass):
  → 어깨 + 엉덩이 중심의 x 이동 방향
  → 도로 방향으로 기울면 '횡단 의도 높음'
```


In [ ]:
def estimate_crossing_intent(keypoints, scores, kp_thresh=0.4):
    """
    보행자 횡단 의도 추정.
    keypoints: (17, 2),  scores: (17,)

    반환:
        intent_score: 0~1 (높을수록 횡단 의도 높음)
        features: 판단 근거 dict
    """
    kp = keypoints
    sc = scores

    features = {}
    scores_list = []

    # ── 1. 몸통 방향 ──
    # 어깨 너비 대비 어깨 깊이 차이 (투영된 이미지에서)
    if sc[5] > kp_thresh and sc[6] > kp_thresh:
        shoulder_w = abs(kp[6, 0] - kp[5, 0])  # x 차이
        # 어깨 너비가 좁으면 = 카메라를 향하거나 등을 보임 (옆모습)
        # → 옆을 보고 있으면 횡단 방향으로 이동 중일 가능성 높음
        img_w_est = 640
        torso_score = 1.0 - min(shoulder_w / (img_w_est * 0.15), 1.0)
        features['torso_orientation'] = round(torso_score, 3)
        features['shoulder_width_px'] = round(shoulder_w, 1)
        scores_list.append(torso_score)

    # ── 2. 보폭 (발목 거리) ──
    if sc[15] > kp_thresh and sc[16] > kp_thresh:
        ankle_dist = np.sqrt((kp[15,0]-kp[16,0])**2 + (kp[15,1]-kp[16,1])**2)
        # 발목이 벌어져 있으면 걷는 중
        stride_score = min(ankle_dist / 80.0, 1.0)
        features['stride_score'] = round(stride_score, 3)
        features['ankle_dist_px'] = round(ankle_dist, 1)
        scores_list.append(stride_score)

    # ── 3. 머리-발 정렬 (수직 자세 = 서있음) ──
    if sc[0] > kp_thresh and (sc[15] > kp_thresh or sc[16] > kp_thresh):
        nose_x = kp[0, 0]
        ankle_x = kp[15, 0] if sc[15] > kp_thresh else kp[16, 0]
        lean = abs(nose_x - ankle_x)
        lean_score = min(lean / 50.0, 1.0)  # 기울수록 이동 중
        features['lean_score'] = round(lean_score, 3)
        scores_list.append(lean_score)

    intent_score = np.mean(scores_list) if scores_list else 0.0
    return round(intent_score, 3), features


# 추론 결과에서 보행자 의도 분석
print('보행자 횡단 의도 분석')
print('=' * 50)

analyzed = 0
for img_path in person_imgs[:5]:
    result = pose_model(img_path, conf=0.3, verbose=False)[0]
    if result.keypoints is None or len(result.keypoints) == 0:
        continue

    kps_all    = result.keypoints.xy.cpu().numpy()
    scores_all = result.keypoints.conf.cpu().numpy()

    for pi in range(len(kps_all)):
        intent, feats = estimate_crossing_intent(kps_all[pi], scores_all[pi])
        bar = '█' * int(intent * 10)
        print(f'{img_path.name} / 사람{pi}: 횡단의도={intent:.2f} {bar}')
        for k, v in feats.items():
            print(f'    {k}: {v}')
        analyzed += 1
        if analyzed >= 6:
            break
    if analyzed >= 6:
        break
print()
print('※ 실제 시스템은 연속 프레임의 keypoint 이동 궤적으로 판단 (시계열 분석)')


## 5. mAP@OKS 평가 파이프라인

Detection의 mAP@IoU와 동일 구조, IoU 자리에 OKS 사용:

```
mAP@OKS = mAP 계산 시 TP 판정 기준을 IoU → OKS로 교체

COCO 평가 기준:
  AP@0.50      : OKS ≥ 0.5 이면 TP
  AP@0.75      : OKS ≥ 0.75 이면 TP (엄격)
  AP@0.50:0.95 : OKS 0.5~0.95 평균 (COCO 메인 지표)
```


In [ ]:
def compute_map_oks(pred_results, gt_data, oks_thresholds=None):
    """
    mAP@OKS 계산 (단순화 버전).
    pred_results: [{'keypoints':(17,2), 'score':float, 'bbox_area':float}, ...]
    gt_data:      [{'keypoints':(17,3), 'bbox_area':float}, ...]
    """
    if oks_thresholds is None:
        oks_thresholds = np.arange(0.5, 1.0, 0.05)

    ap_list = []

    for thr in oks_thresholds:
        # 각 threshold에서 AP 계산
        tp_list = []
        matched_gt = set()

        # score 높은 순 정렬
        sorted_preds = sorted(pred_results, key=lambda x: x['score'], reverse=True)

        for pred in sorted_preds:
            best_oks, best_gi = 0.0, -1
            for gi, gt in enumerate(gt_data):
                if gi in matched_gt:
                    continue
                oks = compute_oks(pred['keypoints'], gt['keypoints'], gt['bbox_area'])
                if oks > best_oks:
                    best_oks, best_gi = oks, gi

            if best_oks >= thr and best_gi >= 0:
                tp_list.append(1)
                matched_gt.add(best_gi)
            else:
                tp_list.append(0)

        # PR 곡선 → AP
        tp_arr = np.array(tp_list)
        cum_tp = np.cumsum(tp_arr)
        n_gt   = len(gt_data)
        prec   = cum_tp / (np.arange(len(tp_arr)) + 1)
        rec    = cum_tp / max(n_gt, 1)

        # 101-point interpolation
        ap = 0.0
        for r_thr in np.linspace(0, 1, 101):
            p_at_r = prec[rec >= r_thr].max() if np.any(rec >= r_thr) else 0.0
            ap += p_at_r / 101
        ap_list.append(ap)

    return {
        'AP@0.50':      round(ap_list[0], 4),
        'AP@0.75':      round(ap_list[5], 4) if len(ap_list) > 5 else None,
        'AP@0.50:0.95': round(np.mean(ap_list), 4),
    }


# ── 시뮬레이션 데이터로 검증 ──
np.random.seed(7)
n_persons = 20

gt_data = []
pred_results = []

for _ in range(n_persons):
    base = np.random.uniform(50, 500, (17, 2))
    vis  = np.random.choice([1, 2], size=(17, 1), p=[0.2, 0.8])
    gt_kp = np.hstack([base, vis])
    gt_data.append({'keypoints': gt_kp, 'bbox_area': 100 * 300})

    noise = np.random.uniform(-15, 15, (17, 2))
    pred_results.append({
        'keypoints': base + noise,
        'score': np.random.uniform(0.6, 0.99),
        'bbox_area': 100 * 300,
    })

metrics = compute_map_oks(pred_results, gt_data)

print('mAP@OKS 평가 결과 (시뮬레이션)')
print('=' * 40)
for k, v in metrics.items():
    print(f'  {k:15s}: {v}')
print()
print('[COCO val2017 YOLOv8n-pose 참고값]')
print('  AP@0.50:0.95 ≈ 0.500')
print('  AP@0.50      ≈ 0.797')
print('  AP@0.75      ≈ 0.544')


## 6. 최종 결과 요약 및 Phase 3 연결


In [ ]:
print('=' * 60)
print('Phase 2 Pose Estimation 파이프라인 최종 결과')
print('=' * 60)
print()
print('[구현 완료]')
print('  1. Gaussian Heatmap   — keypoint 확률 분포 직접 생성')
print('  2. OKS                — keypoint용 IoU 직접 구현')
print('  3. YOLOv8n-pose       — 추론 + Skeleton 시각화')
print('  4. 보행자 의도 예측   — 몸통방향·보폭·기울기 분석')
print('  5. mAP@OKS            — AP@0.5 / @0.75 / @0.5:0.95')
print()
print('[평가 지표]')
for k, v in metrics.items():
    print(f'  {k:15s}: {v}  (시뮬레이션)')
print()
print('[Phase 1~2 재사용 확인]')
print('  IoU 원리    → OKS로 확장 (거리 기반 유사도) ✓')
print('  mAP 파이프라인 → mAP@OKS로 확장 ✓')
print('  ByteTrack   → Pose Track으로 확장 가능 ✓')
print()
print('[Phase 3 BEV 연결 고리]')
print('  2D keypoint (x, y)    → BEV에서 (X, Y, Z) 3D로 확장')
print('  보행자 방향 추정      → BEV에서 heading angle로 활용')
print('  Heatmap 개념          → BEV feature map 생성과 동일 구조')
